# Clinical Severity Correlation — HbT Activation vs Anxiety Scores

**Goal:** Test whether prefrontal HbT STD amplitude (per channel, per task) correlates with
clinical anxiety severity in GAD subjects.

**Measures:**
- **STAI-T** (Trait Anxiety Inventory — Trait scale): all 16 GAD subjects
- **HAMA** (Hamilton Anxiety Rating Scale): 15 GAD subjects (LA063 not administered, HAMA_sum=0)

**Method:** Spearman rank correlation (non-parametric; small N, potential outliers)
followed by Benjamini-Hochberg FDR correction across 23 channels.

**Note on directionality:** Higher STD amplitude = stronger hemodynamic fluctuation.
Positive r → more severe anxiety associated with higher activation.

## 1. Imports & Configuration

In [1]:
import os
import json
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

PROJECT_ROOT  = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '..', '..', '..'))
DATA_DIR      = os.path.join(PROJECT_ROOT, 'data', 'processed-no-grid')
SUBJECTS_JSON = os.path.join(PROJECT_ROOT, 'data', 'subjects.json')
SCORES_XLSX   = os.path.join(PROJECT_ROOT, 'research', 'Anxiety Score Scale_Cleaned.xlsx')
FIG_DIR       = os.path.join(PROJECT_ROOT, 'src', 'notebook', 'statistical-analysis/04_severity_correlation')

TASKS = ['GNG', '1backWM', 'VF', 'SS']

print(f'Project root : {PROJECT_ROOT}')
print(f'Tasks        : {TASKS}')

Project root : /home/user/jeffrymahbuubi/PROJECTS/1-fNIRS-Grid-Base-Method
Tasks        : ['GNG', '1backWM', 'VF', 'SS']


## 2. Channel Mapping

In [2]:
CHANNEL_NAMES = ['S1_D1','S1_D3','S2_D2','S2_D1','S2_D5','S3_D1','S3_D3','S3_D4','S3_D6',
                 'S4_D4','S4_D5','S4_D7','S5_D2','S5_D5','S5_D8','S6_D3','S6_D6',
                 'S7_D4','S7_D6','S7_D7','S8_D5','S8_D7','S8_D8']
GRID_POS = [(0,2),(1,1),(0,4),(0,3),(1,4),(1,2),(2,1),(2,2),(3,1),(2,3),(2,4),(3,4),
            (1,5),(2,5),(3,6),(3,0),(4,1),(3,2),(4,2),(4,3),(3,5),(4,4),(4,5)]
N_CH = len(CHANNEL_NAMES)
print(f'{N_CH} channels on 5\u00d77 grid.')

23 channels on 5×7 grid.


## 3. Load Severity Scores

In [3]:
with open(SUBJECTS_JSON) as f:
    subj = json.load(f)
gad_list = subj['anxiety']  # n=16

df_scores = pd.read_excel(SCORES_XLSX).set_index('Subject')

scores = {}
for sid in gad_list:
    row = df_scores.loc[sid]
    scores[sid] = {'STAIT': float(row['STAIT_sum']), 'HAMA': float(row['HAMA_sum'])}

stait_all = np.array([scores[s]['STAIT'] for s in gad_list])   # (16,)
hama_all  = np.array([scores[s]['HAMA']  for s in gad_list])   # (16,); LA063=0

# Exclude LA063 from HAMA (not administered)
hama_mask = hama_all > 0
gad_hama  = [s for s, m in zip(gad_list, hama_mask) if m]  # n=15
hama_vals = hama_all[hama_mask]

print(f'STAI-T : n={len(stait_all)}, mean={stait_all.mean():.1f} \u00b1 {stait_all.std():.1f},'
      f' range=[{stait_all.min():.0f}, {stait_all.max():.0f}]')
print(f'HAMA   : n={len(hama_vals)}, mean={hama_vals.mean():.1f} \u00b1 {hama_vals.std():.1f},'
      f' range=[{hama_vals.min():.0f}, {hama_vals.max():.0f}]')
print()
print(f'{"Subject":10s}  {"STAI-T":>8s}  {"HAMA":>6s}')
for sid in gad_list:
    hama_str = f"{scores[sid]['HAMA']:.0f}" if scores[sid]['HAMA'] > 0 else 'N/A'
    print(f'  {sid:10s}  {scores[sid]["STAIT"]:>8.0f}  {hama_str:>6s}')

STAI-T : n=16, mean=57.2 ± 6.6, range=[45, 68]
HAMA   : n=15, mean=23.5 ± 8.5, range=[11, 40]

Subject       STAI-T    HAMA
  AA013             45      14
  AA041             57      33
  AA056             60      27
  AA064             51      13
  EA055             55      40
  EA060             62      18
  EA061             50      21
  EA062             58      11
  LA042             55      19
  LA051             65      26
  LA052             67      28
  LA054             47      16
  LA057             55      31
  LA058             57      20
  LA059             68      35
  LA063             63     N/A


## 4. Load HbT Activations (GAD Subjects)

In [4]:
def load_activation(subject_id, task, group):
    path = os.path.join(DATA_DIR, task, group, subject_id, 'hbt')
    concat = np.concatenate(
        [np.load(os.path.join(path, f)) for f in sorted(os.listdir(path))], axis=1
    )
    return concat.std(axis=1)  # (23,)

act = {}  # act[task] → (16, 23)
for task in TASKS:
    act[task] = np.array([load_activation(sid, task, 'anxiety') for sid in gad_list])

act_mean = np.mean([act[t] for t in TASKS], axis=0)  # (16, 23)

print('Activation loaded.')
for task in TASKS:
    print(f'  {task:10s}: shape={act[task].shape}, mean={act[task].mean():.4f}')
print(f'  Mean across tasks: shape={act_mean.shape}')

Activation loaded.
  GNG       : shape=(16, 23), mean=0.9192
  1backWM   : shape=(16, 23), mean=0.9295
  VF        : shape=(16, 23), mean=0.9397
  SS        : shape=(16, 23), mean=0.9415
  Mean across tasks: shape=(16, 23)


## 5. Spearman Correlations per Channel

In [5]:
def spearman_per_ch(activation, scores_vec):
    """Return r, p, p_fdr arrays of length N_CH."""
    r_arr = np.zeros(N_CH)
    p_arr = np.zeros(N_CH)
    for ch in range(N_CH):
        r, p = stats.spearmanr(activation[:, ch], scores_vec)
        r_arr[ch] = r
        p_arr[ch] = p
    _, p_fdr, _, _ = multipletests(p_arr, method='fdr_bh')
    return r_arr, p_arr, p_fdr

results = {}

# STAI-T (n=16)
for task in TASKS:
    r, p, p_fdr = spearman_per_ch(act[task], stait_all)
    results[f'STAIT_{task}'] = {'r': r, 'p': p, 'p_fdr': p_fdr}
r, p, p_fdr = spearman_per_ch(act_mean, stait_all)
results['STAIT_Mean'] = {'r': r, 'p': p, 'p_fdr': p_fdr}

# HAMA (n=15; exclude LA063)
for task in TASKS:
    r, p, p_fdr = spearman_per_ch(act[task][hama_mask], hama_vals)
    results[f'HAMA_{task}'] = {'r': r, 'p': p, 'p_fdr': p_fdr}
r, p, p_fdr = spearman_per_ch(act_mean[hama_mask], hama_vals)
results['HAMA_Mean'] = {'r': r, 'p': p, 'p_fdr': p_fdr}

print(f'{"Key":18s}  {"sig(p<.05)":>10s}  {"sig(FDR)":>8s}  {"top ch":>8s}  {"r":>7s}  {"p":>8s}')
print('-' * 72)
for key, res in results.items():
    sig_raw = (res['p'] < 0.05).sum()
    sig_fdr = (res['p_fdr'] < 0.05).sum()
    top_idx = int(np.argmax(np.abs(res['r'])))
    print(f'  {key:16s}  {sig_raw:>10d}  {sig_fdr:>8d}  '
          f'{CHANNEL_NAMES[top_idx]:>8s}  {res["r"][top_idx]:>7.3f}  {res["p"][top_idx]:>8.4f}')

Key                 sig(p<.05)  sig(FDR)    top ch        r         p
------------------------------------------------------------------------
  STAIT_GNG                  1         0     S8_D8   -0.546    0.0286
  STAIT_1backWM              0         0     S5_D2   -0.458    0.0747
  STAIT_VF                   0         0     S7_D4    0.294    0.2695
  STAIT_SS                   0         0     S5_D2   -0.280    0.2928
  STAIT_Mean                 0         0     S5_D2   -0.342    0.1942
  HAMA_GNG                   0         0     S8_D8   -0.461    0.0839
  HAMA_1backWM               0         0     S4_D7   -0.446    0.0953
  HAMA_VF                    0         0     S1_D3    0.339    0.2160
  HAMA_SS                    0         0     S6_D3   -0.443    0.0983
  HAMA_Mean                  0         0     S6_D6   -0.439    0.1014


## 6. Visualizations

In [6]:
# ── Figure 1: Severity score distributions ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Anxiety Severity Score Distributions (GAD Subjects)', fontsize=12)

ax = axes[0]
ax.hist(stait_all, bins=8, color='#5577CC', edgecolor='white', alpha=0.85)
ax.axvline(stait_all.mean(), color='navy', lw=2, ls='--',
           label=f'Mean={stait_all.mean():.1f}\nSD={stait_all.std():.1f}')
ax.set_xlabel('STAI-T Score')
ax.set_ylabel('Count')
ax.set_title(f'STAI-T (n={len(stait_all)})')
ax.legend(fontsize=9)

ax = axes[1]
ax.hist(hama_vals, bins=7, color='#CC5577', edgecolor='white', alpha=0.85)
ax.axvline(hama_vals.mean(), color='darkred', lw=2, ls='--',
           label=f'Mean={hama_vals.mean():.1f}\nSD={hama_vals.std():.1f}')
ax.set_xlabel('HAMA Score')
ax.set_title(f'HAMA (n={len(hama_vals)}, LA063 excluded)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_severity_score_dist.png'), dpi=150, bbox_inches='tight')
plt.close()
print('Saved: fig_severity_score_dist.png')

Saved: fig_severity_score_dist.png


In [7]:
# ── Shared topo helper ───────────────────────────────────────────────────────
def plot_topo_r(ax, r_vals, p_vals, title, vmin=-0.7, vmax=0.7):
    grid = np.full((5, 7), np.nan)
    for ch_i, (row, col) in enumerate(GRID_POS):
        grid[row, col] = r_vals[ch_i]
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    im = ax.imshow(grid, cmap='RdBu_r', norm=norm, aspect='auto')
    for ch_i, (row, col) in enumerate(GRID_POS):
        r_v = r_vals[ch_i]
        p_v = p_vals[ch_i]
        txt_c = 'white' if abs(r_v) > 0.45 else 'black'
        marker = '*' if p_v < 0.05 else ''
        ax.text(col, row, f'{r_v:.2f}{marker}', ha='center', va='center',
                fontsize=6.5, color=txt_c,
                fontweight='bold' if p_v < 0.05 else 'normal')
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    return im

# ── Figure 2: STAI-T topographic maps ───────────────────────────────────────
panel_labels = TASKS + ['Mean']
stait_keys   = [f'STAIT_{t}' for t in TASKS] + ['STAIT_Mean']

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Spearman r: HbT STD Activation vs STAI-T (GAD, n=16)\n(* p<0.05 uncorrected; red=positive)',
             fontsize=11)
for ax, label, key in zip(axes, panel_labels, stait_keys):
    im = plot_topo_r(ax, results[key]['r'], results[key]['p'], label)
plt.colorbar(im, ax=axes[-1], label='Spearman r', shrink=0.8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_stait_correlation_topo.png'), dpi=150, bbox_inches='tight')
plt.close()
print('Saved: fig_stait_correlation_topo.png')

Saved: fig_stait_correlation_topo.png


In [8]:
# ── Figure 3: HAMA topographic maps ─────────────────────────────────────────
hama_keys = [f'HAMA_{t}' for t in TASKS] + ['HAMA_Mean']

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Spearman r: HbT STD Activation vs HAMA (GAD, n=15, LA063 excluded)\n(* p<0.05 uncorrected)',
             fontsize=11)
for ax, label, key in zip(axes, panel_labels, hama_keys):
    im = plot_topo_r(ax, results[key]['r'], results[key]['p'], label)
plt.colorbar(im, ax=axes[-1], label='Spearman r', shrink=0.8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_hama_correlation_topo.png'), dpi=150, bbox_inches='tight')
plt.close()
print('Saved: fig_hama_correlation_topo.png')

Saved: fig_hama_correlation_topo.png


In [9]:
# ── Figure 4: S7_D6 scatter plots (per task × score) ────────────────────────
ch_idx_s7d6 = CHANNEL_NAMES.index('S7_D6')  # = 18

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('S7_D6 HbT STD Amplitude vs Clinical Severity (GAD Subjects)', fontsize=12)

for col_i, task in enumerate(TASKS):
    # ── STAI-T row ──
    ax = axes[0, col_i]
    x = act[task][:, ch_idx_s7d6]
    y = stait_all
    r_val, p_val = stats.spearmanr(x, y)
    ax.scatter(x, y, color='#5577CC', s=55, alpha=0.75, edgecolors='navy', lw=0.5)
    xline = np.linspace(x.min(), x.max(), 100)
    m, b = np.polyfit(x, y, 1)
    ax.plot(xline, m * xline + b, color='navy', lw=1.5, ls='--')
    ax.set_xlabel(f'S7_D6 STD ({task})', fontsize=9)
    if col_i == 0:
        ax.set_ylabel('STAI-T Score', fontsize=10)
    sig_str = '*' if p_val < 0.05 else ''
    ax.set_title(f'{task}: r={r_val:.3f}, p={p_val:.3f}{sig_str}', fontsize=9,
                 color='darkred' if p_val < 0.05 else 'black')

    # ── HAMA row ──
    ax = axes[1, col_i]
    x_h = act[task][hama_mask, ch_idx_s7d6]
    y_h = hama_vals
    r_val, p_val = stats.spearmanr(x_h, y_h)
    ax.scatter(x_h, y_h, color='#CC5577', s=55, alpha=0.75, edgecolors='darkred', lw=0.5)
    xline = np.linspace(x_h.min(), x_h.max(), 100)
    m, b = np.polyfit(x_h, y_h, 1)
    ax.plot(xline, m * xline + b, color='darkred', lw=1.5, ls='--')
    ax.set_xlabel(f'S7_D6 STD ({task})', fontsize=9)
    if col_i == 0:
        ax.set_ylabel('HAMA Score', fontsize=10)
    sig_str = '*' if p_val < 0.05 else ''
    ax.set_title(f'{task}: r={r_val:.3f}, p={p_val:.3f}{sig_str}', fontsize=9,
                 color='darkred' if p_val < 0.05 else 'black')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_s7d6_severity_scatter.png'), dpi=150, bbox_inches='tight')
plt.close()
print('Saved: fig_s7d6_severity_scatter.png')

Saved: fig_s7d6_severity_scatter.png


## 7. Top Significant Channels Summary

In [10]:
# ── Collect all p<0.05 channel-score-task combinations ──────────────────────
rows = []
for key, res in results.items():
    score, task = key.split('_', 1)  # e.g. 'STAIT', 'GNG'
    n = len(stait_all) if score == 'STAIT' else len(hama_vals)
    for ch_i in range(N_CH):
        rows.append({
            'score': score, 'task': task, 'channel': CHANNEL_NAMES[ch_i],
            'ch_idx': ch_i, 'n': n,
            'r': res['r'][ch_i], 'p': res['p'][ch_i], 'p_fdr': res['p_fdr'][ch_i],
            'sig_raw': res['p'][ch_i] < 0.05, 'sig_fdr': res['p_fdr'][ch_i] < 0.05
        })
df_all = pd.DataFrame(rows)

df_sig = df_all[df_all['sig_raw']].sort_values('p')

print('=== Significant channels (p<0.05 uncorrected) ===')
if len(df_sig) == 0:
    print('  None reached p<0.05 uncorrected.')
else:
    print(df_sig[['score','task','channel','n','r','p','p_fdr','sig_fdr']].to_string(index=False))

print()
df_fdr = df_all[df_all['sig_fdr']].sort_values('p_fdr')
print('=== Significant channels (FDR q<0.05) ===')
if len(df_fdr) == 0:
    print('  None survived FDR correction.')
else:
    print(df_fdr[['score','task','channel','n','r','p','p_fdr']].to_string(index=False))

print()
print('=== S7_D6 summary (primary channel) ===')
df_s7d6 = df_all[df_all['channel'] == 'S7_D6'][['score','task','r','p','p_fdr','sig_raw']]
print(df_s7d6.to_string(index=False))

=== Significant channels (p<0.05 uncorrected) ===
score task channel  n         r        p    p_fdr  sig_fdr
STAIT  GNG   S8_D8 16 -0.546129 0.028625 0.572939    False

=== Significant channels (FDR q<0.05) ===
  None survived FDR correction.

=== S7_D6 summary (primary channel) ===
score    task         r        p    p_fdr  sig_raw
STAIT     GNG  0.157935 0.559099 0.769522    False
STAIT 1backWM -0.039853 0.883498 0.967640    False
STAIT      VF  0.143174 0.596819 0.958255    False
STAIT      SS  0.053137 0.845046 0.972338    False
STAIT    Mean -0.004428 0.987015 0.987015    False
 HAMA     GNG -0.178571 0.524284 0.875534    False
 HAMA 1backWM  0.028571 0.919490 1.000000    False
 HAMA      VF -0.228571 0.412566 0.959700    False
 HAMA      SS -0.317857 0.248291 0.952300    False
 HAMA    Mean -0.217857 0.435393 0.791019    False


In [11]:
# ── Figure 5: Top channels ranked by |r| (mean task) ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Top 10 Channels by Mean |r| Across Tasks', fontsize=12)

for ax_i, score in enumerate(['STAIT', 'HAMA']):
    score_keys = [f'{score}_{t}' for t in TASKS]
    mean_abs_r = np.mean([np.abs(results[k]['r']) for k in score_keys], axis=0)  # (23,)
    top10_idx  = np.argsort(mean_abs_r)[::-1][:10]

    ax = axes[ax_i]
    colors_bar = []
    for ch_i in top10_idx:
        mean_r = np.mean([results[k]['r'][ch_i] for k in score_keys])
        colors_bar.append('#5577CC' if mean_r >= 0 else '#CC5577')

    bars = ax.barh([CHANNEL_NAMES[i] for i in top10_idx],
                   mean_abs_r[top10_idx], color=colors_bar, edgecolor='white', alpha=0.85)
    ax.axvline(0.5, color='gray', lw=1, ls='--', label='|r|=0.5 (medium)')
    ax.set_xlabel('Mean |Spearman r| across 4 tasks')
    n_lbl = 16 if score == 'STAIT' else 15
    ax.set_title(f'{score} (n={n_lbl})')
    ax.legend(fontsize=8)
    ax.invert_yaxis()

pos_patch = mpatches.Patch(color='#5577CC', label='Mean r ≥ 0 (activation ↑ → severity ↑)')
neg_patch = mpatches.Patch(color='#CC5577', label='Mean r < 0 (activation ↑ → severity ↓)')
fig.legend(handles=[pos_patch, neg_patch], loc='lower center', ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_severity_top_channels.png'), dpi=150, bbox_inches='tight')
plt.close()
print('Saved: fig_severity_top_channels.png')

Saved: fig_severity_top_channels.png


## 8. Paper-Ready Summary

In [12]:
s7d6_stait = df_all[(df_all['channel'] == 'S7_D6') & (df_all['score'] == 'STAIT')]
s7d6_hama  = df_all[(df_all['channel'] == 'S7_D6') & (df_all['score'] == 'HAMA')]

print('=' * 70)
print('  CLINICAL SEVERITY CORRELATION — SUMMARY')
print('=' * 70)
print(f'Measure     : HbT STD amplitude per channel (prefrontal fNIRS)')
print(f'Test        : Spearman rank correlation + BH-FDR (23 channels)')
print(f'Subjects    : GAD only  (STAIT n=16; HAMA n=15, LA063 excluded)')
print()

for score in ['STAIT', 'HAMA']:
    n = 16 if score == 'STAIT' else 15
    keys = [f'{score}_{t}' for t in TASKS] + [f'{score}_Mean']
    total_sig = sum((results[k]['p'] < 0.05).sum() for k in keys)
    total_fdr = sum((results[k]['p_fdr'] < 0.05).sum() for k in keys)
    print(f'{score} (n={n}): total sig combinations={total_sig} uncorr, {total_fdr} FDR')
    top_r_abs = max(np.max(np.abs(results[k]['r'])) for k in keys)
    for k in keys:
        top_idx = np.argmax(np.abs(results[k]['r']))
        if results[k]['p'][top_idx] < 0.05:
            print(f'  {k:15s}: top={CHANNEL_NAMES[top_idx]} r={results[k]["r"][top_idx]:.3f}'
                  f' p={results[k]["p"][top_idx]:.4f}')
print()

print('S7_D6 correlations:')
print(s7d6_stait[['task','r','p','p_fdr','sig_raw']].to_string(index=False))
print()
print(s7d6_hama[['task','r','p','p_fdr','sig_raw']].to_string(index=False))
print()

print('─── Draft for paper Results ───')
n_sig_stait = sum((results[f'STAIT_{t}']['p'] < 0.05).sum() for t in TASKS)
n_sig_hama  = sum((results[f'HAMA_{t}']['p'] < 0.05).sum() for t in TASKS)
print(f'''  Within the GAD group (n=16), we tested whether per-channel HbT
  STD amplitude correlates with clinical anxiety severity using
  Spearman rank correlations with BH-FDR correction (23 channels).
  Across all four tasks, {n_sig_stait} channel-task combinations showed
  nominally significant correlation with STAI-T scores (p<0.05 uncorrected);
  {n_sig_hama} with HAMA scores (p<0.05 uncorrected, n=15).
  No channel survived FDR correction, likely reflecting limited power
  with N=16 GAD subjects. The most discriminative channel from the
  group-level analysis (S7_D6) showed [describe r direction and magnitude]
  correlations with both STAI-T and HAMA, though these did not reach
  statistical significance after correction, consistent with the moderate
  sample size.''')

  CLINICAL SEVERITY CORRELATION — SUMMARY
Measure     : HbT STD amplitude per channel (prefrontal fNIRS)
Test        : Spearman rank correlation + BH-FDR (23 channels)
Subjects    : GAD only  (STAIT n=16; HAMA n=15, LA063 excluded)

STAIT (n=16): total sig combinations=1 uncorr, 0 FDR
  STAIT_GNG      : top=S8_D8 r=-0.546 p=0.0286
HAMA (n=15): total sig combinations=0 uncorr, 0 FDR

S7_D6 correlations:
   task         r        p    p_fdr  sig_raw
    GNG  0.157935 0.559099 0.769522    False
1backWM -0.039853 0.883498 0.967640    False
     VF  0.143174 0.596819 0.958255    False
     SS  0.053137 0.845046 0.972338    False
   Mean -0.004428 0.987015 0.987015    False

   task         r        p    p_fdr  sig_raw
    GNG -0.178571 0.524284 0.875534    False
1backWM  0.028571 0.919490 1.000000    False
     VF -0.228571 0.412566 0.959700    False
     SS -0.317857 0.248291 0.952300    False
   Mean -0.217857 0.435393 0.791019    False

─── Draft for paper Results ───
  Within the GAD gro

In [13]:
# Export CSV
out_path = os.path.join(FIG_DIR, 'results_severity_correlation.csv')
df_all.to_csv(out_path, index=False)
print(f'Saved: {out_path}  ({df_all.shape})')

generated = [
    'fig_severity_score_dist.png',
    'fig_stait_correlation_topo.png',
    'fig_hama_correlation_topo.png',
    'fig_s7d6_severity_scatter.png',
    'fig_severity_top_channels.png',
]
print('\nGenerated figures:')
for f in generated:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f'  {f:42s}  {status}')

Saved: results_severity_correlation.csv  ((230, 10))

Generated figures:
  fig_severity_score_dist.png                 OK
  fig_stait_correlation_topo.png              OK
  fig_hama_correlation_topo.png               OK
  fig_s7d6_severity_scatter.png               OK
  fig_severity_top_channels.png               OK
